In [1]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Thiết lập đường dẫn dữ liệu
basic_dir = "src/data/forecasts/A_BasicForecaster_most_recent_gpt_5.4_mini"
hybrid_dir = "src/data/forecasts/A_HybridACDForecaster_gpt_5.4_mini"

# Đọc file stats_summary.json
with open(os.path.join(basic_dir, "stats_summary.json"), "r") as f:
    basic_stats = json.load(f)

with open(os.path.join(hybrid_dir, "stats_summary.json"), "r") as f:
    hybrid_stats = json.load(f)

print("Tải dữ liệu thành công!")
print(f"Mô hình Basic Forecaster: {basic_stats.get('forecaster', 'N/A')}")
print(f"Mô hình Hybrid Forecaster: {hybrid_stats.get('forecaster', 'N/A')}")

Tải dữ liệu thành công!
Mô hình Basic Forecaster: BasicForecaster
Mô hình Hybrid Forecaster: HybridACDForecaster


## 1. Phân tích Nhất quán Tổng hợp (Aggregated Consistency Analysis)

Độ vi phạm nhất quán tổng hợp được đo lường qua 3 hệ số:
- **Default (Arbitrage Metric)**: Độ vi phạm chênh lệch giá dựa trên định lý Dutch Book. Biểu thị thiệt hại tài chính tối đa khi đặt cược theo xác suất của mô hình.
- **Frequentist Metric**: Đo lường bằng thống kê tần suất, đánh giá độ lệch dựa trên kiểm định giả thuyết.
- **Default Scaled**: Phiên bản chuẩn hóa của chỉ số Arbitrage nhằm loại bỏ tác động quy mô câu hỏi.

In [ ]:
# Trích xuất dữ liệu tổng hợp
mtypes = ["default", "frequentist", "default_scaled"]
basic_agg = [basic_stats["aggregated"][mtype]["avg_violation"] for mtype in mtypes]
hybrid_agg = [hybrid_stats["aggregated"][mtype]["avg_violation"] for mtype in mtypes]

# Vẽ biểu đồ so sánh
plt.figure(figsize=(10, 6), dpi=100)
x = np.arange(len(mtypes))
width = 0.35

plt.bar(x - width/2, basic_agg, width, label='BasicForecaster', color='#E74C3C', edgecolor='black', alpha=0.9)
plt.bar(x + width/2, hybrid_agg, width, label='HybridACDForecaster', color='#2ECC71', edgecolor='black', alpha=0.9)

plt.ylabel('Độ vi phạm trung bình (Average Violation Magnitude)', fontsize=12, fontweight='bold')
plt.title('Độ vi phạm Nhất quán Tổng hợp: Basic vs HybridACD', fontsize=14, fontweight='bold', pad=15)
plt.xticks(x, ['Default (Arbitrage)', 'Frequentist', 'Default Scaled'], fontsize=11)
plt.legend(frameon=True, facecolor='white', edgecolor='gray')
plt.grid(axis='y', linestyle='--', alpha=0.5)

# Hiển thị giá trị số trên đầu các cột
for i in range(len(mtypes)):
    plt.text(i - width/2, basic_agg[i] + 0.015, f"{basic_agg[i]:.4f}", ha='center', va='bottom', fontweight='bold', color='#C0392B')
    plt.text(i + width/2, hybrid_agg[i] + 0.015, f"{hybrid_agg[i]:.4f}", ha='center', va='bottom', fontweight='bold', color='#27AE60')

plt.ylim(0, max(max(basic_agg), max(hybrid_agg)) * 1.15)
plt.tight_layout()
plt.show()

### Nhận xét & Đánh giá:

- **Default (Arbitrage) Metric**: Giảm từ **0.4909** ở Basic xuống còn **0.0107** ở HybridACD (giảm **97.8%**!). Điều này chứng tỏ HybridACD đã triệt tiêu hầu hết cơ hội "bào tiền" Dutch Book của mô hình.
- **Frequentist Metric**: Giảm từ **1.0423** xuống còn **0.0227** (giảm **97.8%**), chứng tỏ độ lệch niềm tin hệ thống của mô hình đã được chỉnh sửa gần như hoàn hảo về mức ngẫu nhiên.
- **Default Scaled**: Giảm từ **0.1809** xuống còn **0.0030** (giảm **98.3%**).

Kết quả tổng hợp cho thấy phương pháp **HybridACD** mang lại cải tiến vượt bậc về độ nhất quán logic tổng thể của mô hình.

## 2. Phân tích Chi tiết Từng Bộ Kiểm tra (Checker-level Analysis)

Chúng ta sẽ so sánh chi tiết hiệu năng của BasicForecaster và HybridACDForecaster trên 10 bộ kiểm tra logic nhất quán (Checkers):
1. **Negation (Phủ định)**: P(P) + P(~P) = 1
2. **Paraphrase (Cách diễn đạt khác)**: P(P) = P(Q)
3. **Consequence (Hệ quả)**: Nếu P => Q thì P(P) <= P(Q)
4. **AndOr**: P(P) + P(Q) = P(P or Q) + P(P and Q)
5. **And**: max(P(P) + P(Q) - 1, 0) <= P(P and Q) <= min(P(P), P(Q))
6. **Or**: max(P(P), P(Q)) <= P(P or Q) <= min(1, P(P) + P(Q))
7. **But**: P(P or Q) = P(P) + P(~P and Q)
8. **Conditional**: P(P) * P(Q|P) = P(P and Q)
9. **CondCond**: P(P) * P(Q|P) * P(R|P and Q) = P(P and Q and R)
10. **Expected Evidence**: P(P) = P(P|Q)P(Q) + P(P|~Q)(1 - P(Q))

In [3]:
checkers = [
    "NegChecker", "AndChecker", "OrChecker", "AndOrChecker", "ButChecker",
    "CondChecker", "ConsequenceChecker", "ParaphraseChecker", "CondCondChecker",
    "ExpectedEvidenceChecker"
]

rows = []
for checker in checkers:
    b_metrics = basic_stats[checker]["overall"]["default"]
    h_metrics = hybrid_stats[checker]["overall"]["default"]
    
    rows.append({
        "Checker": checker,
        "Basic_Violations": b_metrics["num_violations"],
        "Basic_ViolationRate": b_metrics["num_violations"] / b_metrics["num_samples"] if b_metrics["num_samples"] > 0 else 0.0,
        "Basic_AvgViolation": b_metrics["avg_violation"],
        "Hybrid_Violations": h_metrics["num_violations"],
        "Hybrid_ViolationRate": h_metrics["num_violations"] / h_metrics["num_samples"] if h_metrics["num_samples"] > 0 else 0.0,
        "Hybrid_AvgViolation": h_metrics["avg_violation"]
    })

df_checkers = pd.DataFrame(rows)

# Hiển thị bảng so sánh chi tiết
df_display = df_checkers.copy()
df_display["Basic_ViolationRate"] = df_display["Basic_ViolationRate"].map(lambda v: f"{v*100:.1f}%")
df_display["Hybrid_ViolationRate"] = df_display["Hybrid_ViolationRate"].map(lambda v: f"{v*100:.1f}%")
df_display["Basic_AvgViolation"] = df_display["Basic_AvgViolation"].round(4)
df_display["Hybrid_AvgViolation"] = df_display["Hybrid_AvgViolation"].round(4)

print("=== BẢNG SO SÁNH CHI TIẾT TỪNG CHECKER ===")
display(df_display)

=== BẢNG SO SÁNH CHI TIẾT TỪNG CHECKER ===


,Checker,Basic_Violations,Basic_ViolationRate,Basic_AvgViolation,Hybrid_Violations,Hybrid_ViolationRate,Hybrid_AvgViolation
0,NegChecker,9,90.0%,0.9529,0,0.0%,0.0000
1,AndChecker,3,30.0%,0.2601,0,0.0%,0.0000
2,OrChecker,2,20.0%,0.2930,0,0.0%,0.0000
3,AndOrChecker,4,40.0%,1.0806,0,0.0%,0.0000
4,ButChecker,4,40.0%,0.3479,1,10.0%,0.0441
5,CondChecker,3,30.0%,0.0667,0,0.0%,0.0000
6,ConsequenceChecker,4,40.0%,0.7242,0,0.0%,0.0000
7,ParaphraseChecker,7,70.0%,0.3276,0,0.0%,0.0000
8,CondCondChecker,7,70.0%,0.1497,0,0.0%,0.0000
9,ExpectedEvidenceChecker,3,30.0%,0.7064,2,20.0%,0.0626


In [ ]:
# Vẽ biểu đồ so sánh tỷ lệ vi phạm (Violation Rate) và mức độ vi phạm trung bình (Avg Violation Magnitude)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), dpi=100)
x_chk = np.arange(len(checkers))
width_chk = 0.35

# 1. Biểu đồ tỷ lệ vi phạm (Violation Rate)
ax1.bar(x_chk - width_chk/2, df_checkers["Basic_ViolationRate"], width_chk, label='BasicForecaster', color='#E74C3C', edgecolor='black', alpha=0.9)
ax1.bar(x_chk + width_chk/2, df_checkers["Hybrid_ViolationRate"], width_chk, label='HybridACDForecaster', color='#2ECC71', edgecolor='black', alpha=0.9)
ax1.set_ylabel('Tỷ lệ vi phạm (Violation Rate)', fontsize=11, fontweight='bold')
ax1.set_title('So sánh Tỷ lệ Vi phạm Logic theo Checker', fontsize=13, fontweight='bold', pad=10)
ax1.set_xticks(x_chk)
ax1.set_xticklabels(checkers, rotation=15, ha='right')
ax1.legend()
ax1.grid(axis='y', linestyle='--', alpha=0.5)
ax1.set_ylim(0, 1.15)

# Thêm giá trị phần trăm
for i, val in enumerate(df_checkers["Basic_ViolationRate"]):
    ax1.text(i - width_chk/2, val + 0.02, f"{val*100:.0f}%", ha='center', va='bottom', fontsize=9, color='#C0392B')
for i, val in enumerate(df_checkers["Hybrid_ViolationRate"]):
    ax1.text(i + width_chk/2, val + 0.02, f"{val*100:.0f}%", ha='center', va='bottom', fontsize=9, color='#27AE60')

# 2. Biểu đồ mức độ vi phạm trung bình (Avg Violation Magnitude)
ax2.bar(x_chk - width_chk/2, df_checkers["Basic_AvgViolation"], width_chk, label='BasicForecaster', color='#E74C3C', edgecolor='black', alpha=0.9)
ax2.bar(x_chk + width_chk/2, df_checkers["Hybrid_AvgViolation"], width_chk, label='HybridACDForecaster', color='#2ECC71', edgecolor='black', alpha=0.9)
ax2.set_ylabel('Độ lớn vi phạm trung bình (Avg Magnitude)', fontsize=11, fontweight='bold')
ax2.set_title('So sánh Mức độ Vi phạm Logic Trung bình (Default)', fontsize=13, fontweight='bold', pad=10)
ax2.set_xticks(x_chk)
ax2.set_xticklabels(checkers, rotation=15, ha='right')
ax2.legend()
ax2.grid(axis='y', linestyle='--', alpha=0.5)
ax2.set_ylim(0, max(max(df_checkers["Basic_AvgViolation"]), max(df_checkers["Hybrid_AvgViolation"])) * 1.2)

# Thêm giá trị cụ thể
for i, val in enumerate(df_checkers["Basic_AvgViolation"]):
    ax2.text(i - width_chk/2, val + 0.02, f"{val:.3f}", ha='center', va='bottom', fontsize=9, color='#C0392B')
for i, val in enumerate(df_checkers["Hybrid_AvgViolation"]):
    ax2.text(i + width_chk/2, val + 0.02, f"{val:.3f}", ha='center', va='bottom', fontsize=9, color='#27AE60')

plt.tight_layout()
plt.show()

### Phân tích chi tiết từng quy tắc logic:

1. **Negation (NegChecker)**:
   - *Basic*: Vi phạm **90.0%** mẫu thử với độ lệch cực cao **0.9529** (ví dụ: AI dự đoán P(P) = 0.9 và P(~P) = 0.85, tổng bằng 1.75 thay vì 1.0).
   - *Hybrid*: Đạt **0.0%** vi phạm (Avg = 0.0). Ràng buộc logic phủ định $P(P) + P(\neg P) = 1$ đã được đáp ứng hoàn hảo.

2. **Paraphrase (ParaphraseChecker)**:
   - *Basic*: Vi phạm **70.0%** với độ lớn **0.3276**. AI dự đoán hai câu hỏi cùng nghĩa nhưng lại cho xác suất khác nhau (ví dụ: đổi từ chủ động sang bị động).
   - *Hybrid*: Đạt **0.0%** vi phạm (Avg = 0.0).

3. **AndOr & Consequence (AndOrChecker, ConsequenceChecker)**:
   - *Basic*: Vi phạm lần lượt **40.0%** và **40.0%** với độ lệch lớn (AndOr: 1.0806, Consequence: 0.7242).
   - *Hybrid*: Cả hai đều được đưa về **0.0%** vi phạm.

4. **But & ExpectedEvidence (ButChecker, ExpectedEvidenceChecker)**:
   - *Basic*: Vi phạm **40%** (But) và **30%** (ExpectedEvidence).
   - *Hybrid*: Giảm đáng kể nhưng vẫn còn vết vi phạm nhỏ. **ButChecker** giảm xuống **10.0%** vi phạm (độ lớn 0.0441), **ExpectedEvidenceChecker** giảm xuống **20.0%** vi phạm (độ lớn 0.0626). Đây là 2 checker phức tạp nhất vì But liên kết phủ định giao ($P(P \lor Q) = P(P) + P(\neg P \land Q)$) và ExpectedEvidence liên kết phân rã xác suất kỳ vọng toàn phần. Việc HybridACD cải tạo được 80-90% các vi phạm này đã là một kết quả cực kỳ ấn tượng.

## 3. Hiệu năng Dự đoán Thực tế (Ground Truth Accuracy)

Ground Truth Accuracy đánh giá độ chính xác của dự báo dựa trên kết quả thực tế thu thập được (Yes/No):
- **Brier Score**: Bình phương sai số giữa xác suất dự đoán và nhãn thực tế ($0$ hoặc $1$). Càng gần $0$ càng chính xác.
- **Platt Brier Score**: Brier score sau khi được căn chỉnh bằng phương pháp hồi quy Platt.
- **Calibration Error**: Sai số hiệu chuẩn (chênh lệch giữa độ tự tin dự báo và tần suất chính xác thực tế).

In [5]:
gt_path = os.path.join(basic_dir, "ground_truth_summary.json")
if os.path.exists(gt_path):
    with open(gt_path, "r") as f:
        basic_gt = json.load(f)
    
    print("=== CHỈ SỐ GROUND TRUTH CỦA BASIC FORECASTER ===")
    print(f"Tổng số câu hỏi đánh giá: {basic_gt.get('total_questions', 'N/A')}")
    print(f"Avg Brier Score:         {basic_gt.get('avg_brier_score', 'N/A')}")
    print(f"Avg Platt Brier Score:   {basic_gt.get('avg_platt_brier_score', 'N/A')}")
    print(f"Brier Score Baseline:    {basic_gt.get('tuned_brier_baseline', 'N/A')}")
    print(f"Calibration Error:       {basic_gt.get('calibration_error', 'N/A')}")
    print(f"Uncertainty:             {basic_gt.get('brier_score_decomposition', {}).get('uncertainty', 'N/A')}")
    print(f"Reliability:             {basic_gt.get('brier_score_decomposition', {}).get('reliability', 'N/A')}")
    print(f"Resolution:              {basic_gt.get('brier_score_decomposition', {}).get('resolution', 'N/A')}")
else:
    print("Không tìm thấy dữ liệu Ground Truth cho BasicForecaster ở thư mục chỉ định.")

=== CHỈ SỐ GROUND TRUTH CỦA BASIC FORECASTER ===
Tổng số câu hỏi đánh giá: 3
Avg Brier Score:         0.0
Avg Platt Brier Score:   0.167
Brier Score Baseline:    0.222
Calibration Error:       0.002
Uncertainty:             0.222
Reliability:             0.667
Resolution:              0.222


### Nhận xét về Ground Truth:

1. **Mẫu thử nhỏ**: Đánh giá Ground Truth chỉ chứa 3 câu hỏi thực tế (Metaculus). Số lượng mẫu này quá nhỏ để rút ra kết luận mang tính thống kê sâu về độ chính xác dự báo (Brier Score = 0.0 nghĩa là mô hình dự đoán chính xác tuyệt đối trên cả 3 câu hỏi này).
2. **Sự vắng mặt của Ground Truth ở HybridACDForecaster**: Trong thư mục đánh giá của HybridACD không có file `ground_truth_summary.json`. Điều này xảy ra do mô hình HybridACD trong lượt chạy này chỉ được thực thi đánh giá nhất quán logic (`src/evaluation.py` trên tập câu hỏi Tuples), không chạy script đánh giá độ chính xác thực tế (`src/ground_truth_run.py`). Theo quy trình hệ thống tại `workflow.md`, việc đánh giá độ nhất quán logic hoàn toàn độc lập và không yêu cầu nhãn thực tế tương lai.

## 4. Giải thích cơ chế giúp HybridACD triệt tiêu vi phạm logic

Sự sụt giảm ngoạn mục về mức độ vi phạm nhất quán logic từ **0.4909** xuống còn **0.0107** là nhờ vào cơ chế phòng thủ 3 lớp của **HybridACDForecaster**:

1. **Adversarial Input Agent (Đối kháng đầu vào)**:
   Sử dụng một mô hình phụ để viết lại câu hỏi thô thành các cấu trúc ngữ pháp phức tạp và đa dạng hơn nhưng bảo toàn ý nghĩa toán học. Điều này loại bỏ các phản xạ máy móc của LLM Forecaster đối với cấu trúc câu đơn giản, bắt buộc mô hình phải suy luận logic thực sự.

2. **Chain-of-Thought (Lập luận dạng chuỗi)**:
   Cho phép mô hình lập luận tự do (scratchpad) trước khi đưa ra xác suất dự đoán cuối cùng, giúp AI tự căn chỉnh và nhận ra các mâu thuẫn logic trong lập luận.

3. **Token Constraint Decoding (TCD - Ràng buộc giải mã)**:
   - **Dynamic Bounds**: Theo dõi các câu hỏi đã trả lời trước đó trong cùng một Tuple câu hỏi và tự động tính toán khoảng xác suất hợp lệ toán học `[lower_bound, upper_bound]` cho câu hỏi tiếp theo dựa trên checker tương ứng.
   - **Logit Bias Masking**: Can thiệp trực tiếp vào logits trong quá trình sinh token số của `gpt-5.4-mini` bằng cách áp logit bias âm vô cực (`-100`) cho mọi token số nằm ngoài khoảng an toàn. Điều này bắt buộc mô hình chỉ được sinh xác suất nằm trong khoảng nhất quán logic.
   - **Softmax Fallback (Clipping)**: Lớp chốt chặn cuối cùng sử dụng `np.clip` để cắt các xác suất về khoảng hợp lệ toán học nếu logit bias bị lỗi parser.

## 5. Kết luận
Phương pháp **HybridACD** đã chứng minh tính hiệu quả vượt trội trong việc kiểm soát tính nhất quán logic xác suất của LLM Forecaster, giải quyết triệt để vấn đề vi phạm logic của mô hình cơ bản và tạo ra một hệ thống dự báo đáng tin cậy hơn nhiều.